# P4 · Spatial Identification of Neo-Rural Hotspots
## *RURIMESCAPE — Paper 1, Step 4*

---

**Paper:** Rural Migration and Land Use in Spain — Paper 1  
**Step:** 4 — Spatial identification and classification of neo-rural hotspots  
**Author:** Juan Zotes  
**Last updated:** 2026-04

---

### Context and purpose

Building on the behavioural matrix from p3, this notebook moves from aspatial classification
to explicit spatial analysis. The core question is: **where** are the municipalities showing
demographic growth in Period B (2018–2025), and are they isolated cases or do they form
spatially coherent clusters?

Two demographic profiles receive special attention:

- **Reverters** (`Reverses in B`): municipalities that declined in A but grew in B —
  the primary neo-rural signal of Paper 1.
- **Sustained dynamisers** (`Grows in both`): municipalities that grew in both periods —
  structurally resilient cases such as Negueira de Muñiz (Lugo) that never entered
  demographic decline.

Both profiles are analysed in **parallel pipelines** (Section 3 and Section 4 respectively),
keeping their conceptual distinction explicit throughout.

The analysis covers both **Rural-Remote** and **Rural-Accessible** typologies.
Rural-Accessible is treated as a parallel, comparable stratum: its growing municipalities
are expected to show stronger peri-urbanisation signals, but this is not assumed — it is
tested through the same spatial criteria applied to Rural-Remote.

The `Loses in B` group (grew in A, declined in B) is retained in the output GeoPackage
as a labelled layer for reference but is not a focus of the spatial analysis.

---

### Analytical structure

| Section | Content |
|---------|--------|
| 0 | Environment, paths, constants |
| 1 | Data loading and spatial join |
| 2 | Distance to nearest urban centre (peri-urban proxy) |
| 3 | **Pipeline A — Reverters** (`Reverses in B`) |
| 3.1 | Spatial outlier classification: mean neighbour `var_acum_pct_B` < 0 |
| 3.2 | Connected-component clusters (Queen contiguity, direct contact) |
| 3.3 | Extended neighbourhood clusters (shared-neighbour proximity) |
| 3.4 | LISA — Local Moran's I for statistical validation |
| 3.5 | Summary table and display |
| 4 | **Pipeline B — Sustained dynamisers** (`Grows in both`) |
| 4.1 | Spatial outlier classification |
| 4.2 | Connected-component clusters |
| 4.3 | Extended neighbourhood clusters |
| 4.4 | LISA validation |
| 4.5 | Summary table and display |
| 5 | Combined output: GeoPackage layers and CSV tables |
| 6 | Interpretation notes |

---

### Spatial outlier criterion

A growing municipality (Reverter or Dynamiser) is a **spatial outlier** (genuine neo-rural
hotspot) if the **unweighted mean** of `var_acum_pct_B` across all Queen-contiguous (shared boundary or vertex) neighbours
is **negative**. This captures the isolation signal: the municipality grows while its
immediate context declines.

The unweighted mean is preferred over population-weighted aggregation to avoid masking
the local signal with the demographic size of neighbours. A weighted version is retained
as a column in the output CSV for future sensitivity comparison.

---

### Cluster typology

| Cluster type | Definition | Spatial tool |
|---|---|---|
| **Isolated outlier** | Grows; mean neighbour `var_acum_pct_B` < 0; no contiguous growing neighbours | Spatial outlier criterion |
| **Direct cluster** | Two or more growing municipalities sharing a Queen boundary (shared boundary or vertex) | Connected components (direct contact) |
| **Extended cluster** | Growing municipalities not in direct contact but sharing a common Queen neighbour | Connected components (shared-neighbour graph) |
| **LISA HH** | High-growth municipality surrounded by neighbours with above-average growth (p < 0.05) | Local Moran\'s I |
| **LISA HL** | High-growth municipality surrounded by neighbours with below-average growth (p < 0.05) | Local Moran\'s I |

LISA is applied to the full rural stratum (not only growing municipalities) to capture
the statistical significance of the spatial pattern.

---

### Peri-urban proxy

The Goerlich (2016) typology incorporates accessibility by construction, so Rural-Remote
municipalities already represent low-accessibility areas. Within Rural-Remote, a Euclidean
distance proxy to the nearest urban centre centroid (derived from `tipo_goerlich` in the
geographic hierarchy GeoPackage) is computed as a supplementary variable.

A threshold of **60 km** is used as an indicative peri-urban boundary, stored as a binary
flag `within_60km_urban`. This threshold is a parameter (`PERIURBAN_KM`) adjustable at
the top of the notebook. The Euclidean approximation is an acknowledged simplification;
road-network travel time (e.g. via OSMnx or OSRM) would improve precision in future work
(see Goerlich et al., correspondence March 2026).

---

### Inputs

| File | Location | Description |
|------|----------|-------------|
| `p2_periods_AB.gpkg` | `data/spatial/processed/` | Municipality polygons with period A and B indicators (2 layers) |
| `p3_behavioural_matrix.csv` | `data/demography/derived/` | Behavioural group per municipality |
| `p0_municipios_goerlich.gpkg` | `data/spatial/processed/` | Full geographic hierarchy including `tipo_goerlich` |

### Outputs

| File | Location | Description |
|------|----------|-------------|
| `p4_spatial_hotspots.gpkg` | `data/spatial/processed/` | Multi-layer GeoPackage (see below) |
| `p4_reverters_classified.csv` | `data/demography/derived/` | Reverters with spatial classification |
| `p4_dynamisers_classified.csv` | `data/demography/derived/` | Sustained dynamisers with spatial classification |
| `p4_cluster_summary.csv` | `data/demography/derived/` | Cluster-level summary table |

**GeoPackage layers (`p4_spatial_hotspots.gpkg`):**

| Layer | Content |
|-------|---------|
| `rural_remote_all` | All Rural-Remote municipalities with full p4 attributes |
| `rural_accessible_all` | All Rural-Accessible municipalities with full p4 attributes |
| `reverters_classified` | Reverters only, both typologies, with outlier/cluster flags |
| `dynamisers_classified` | Sustained dynamisers only, both typologies, with outlier/cluster flags |
| `lisa_results_rural` | LISA quadrant and p-value for all rural municipalities |

All layers in EPSG:25830 (ETRS89 / UTM zone 30N).

---

---
## 0 · Environment, paths, constants

In [ ]:
"""
Notebook  : p4_spatial_hotspots.ipynb
Author    : Juan Zotes
Created   : 2026-04

Purpose:
    Spatial identification and classification of neo-rural hotspots in Spain.
    Two parallel pipelines:
        A. Reverters (Reverses in B): municipalities crossing from decline to growth
           after T=2018 — primary neo-rural signal of Paper 1.
        B. Sustained dynamisers (Grows in both): structurally resilient municipalities
           that never entered demographic decline.

    Both pipelines applied to Rural-Remote and Rural-Accessible in parallel.

    Spatial methods:
        1. Spatial outlier criterion: grows while mean neighbour var_acum_pct_B < 0
        2. Connected-component clusters (Queen contiguity, direct contact)
        3. Extended neighbourhood clusters (shared-neighbour graph)
        4. LISA Local Moran's I (full rural stratum)

Inputs:
    - p2_periods_AB.gpkg                        (spatial/processed)
    - p3_behavioural_matrix.csv                 (demography/derived)
    - p0_municipios_goerlich.gpkg               (spatial/processed)

Outputs:
    - p4_spatial_hotspots.gpkg                  (spatial/processed)   5 layers
    - p4_reverters_classified.csv               (demography/derived)
    - p4_dynamisers_classified.csv              (demography/derived)
    - p4_cluster_summary.csv                    (demography/derived)

Notes:
    - Neighbour mean: unweighted var_acum_pct_B; weighted version retained as column.
    - Peri-urban proxy: Euclidean distance to nearest urban or intermediate centre
      (PERIURBAN_KM = 60). Produces dist_nearest_urban_km, dist_nearest_intermediate_km,
      and urban_proximity categorical column ('near_urban' / 'near_intermediate' / 'remote').
    - Queen contiguity (shared boundary or vertex) throughout; EPSG:25830 for all spatial operations.
    - LISA requires pysal / esda: pip install esda libpysal
"""

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

# LISA / spatial weights
try:
    from libpysal.weights import Queen as QueenW
    from esda.moran import Moran, Moran_Local
    LISA_AVAILABLE = True
    print('esda / libpysal loaded — LISA analysis enabled.')
except ImportError:
    LISA_AVAILABLE = False
    print('WARNING: esda / libpysal not found. Run: pip install esda libpysal')
    print('LISA section will be skipped; all other analysis will run normally.')

print('Libraries loaded.')

In [ ]:
# ── Root: adjust if project moves ────────────────────────────────────────────
ROOT = Path(r'C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain')

# ── Input paths ───────────────────────────────────────────────────────────────
GPKG_P2        = ROOT / 'data/spatial/processed/p2_periods_AB.gpkg'
CSV_MATRIX     = ROOT / 'data/demography/derived/paper1/p3_behavioural_matrix.csv'
GPKG_P0_GOERLICH = ROOT / 'data/spatial/processed/p0_municipios_goerlich.gpkg'

# ── Output paths ──────────────────────────────────────────────────────────────
SPATIAL_PROC   = ROOT / 'data/spatial/processed'
DEMO_DERIV     = ROOT / 'data/demography/derived/paper1'
FIG_DIR        = ROOT / 'figures/p4_spatial_hotspots'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Parameters ────────────────────────────────────────────────────────────────
PERIURBAN_KM   = 60          # Euclidean distance threshold for peri-urban flag
CRS_METRIC     = 'EPSG:25830'  # ETRS89 / UTM zone 30N — for distance calculations
LISA_PERMUT    = 9999          # Permutations for LISA significance
LISA_ALPHA     = 0.05          # Significance threshold

# ── Typology constants (consistent with p2/p3) ────────────────────────────────
RURAL_TYPES    = ['Rural - Remoto', 'Rural - Accesible']
URBAN_TYPES        = ['Urbano - Cerrado', 'Urbano - Abierto']
INTERMEDIATE_TYPES = ['Intermedio - Cerrado', 'Intermedio - Abierto']
TARGET_GROUPS  = ['Reverses in B', 'Grows in both']   # Primary focus groups

# --- Behavioural group colours — consistent with choropleth map palette
# Blue = population loss, Red = population gain
GROUP_COLORS = {
    'Grows in both'          : '#d7191c',  # dark red    — sustained growth
    'Reverses in B'          : '#fdae61',  # light orange — reverter post-inflection
    'Loses in B'             : '#abd9e9',  # light blue  — lost what was gained
    'Structural depopulation': '#2c7bb6',  # dark blue   — persistent decline
}

print('Paths and parameters defined.')
for p in [GPKG_P2, CSV_MATRIX, GPKG_P0_GOERLICH]:
    status = '✓' if p.exists() else '✗  NOT FOUND'
    print(f'  {status}  {p.name}')

---
## 1 · Data loading and spatial join

In [ ]:
# ── Load behavioural matrix ───────────────────────────────────────────────────
df_matrix = pd.read_csv(CSV_MATRIX, sep=';', encoding='utf-8-sig', dtype={'Mun_Code': str})
df_matrix['Mun_Code'] = df_matrix['Mun_Code'].str.zfill(5)
print(f'Behavioural matrix loaded: {len(df_matrix):,} municipalities')
print(f'Columns: {list(df_matrix.columns)}')

In [ ]:
# ── Load geometry from p2 GeoPackage (Period B layer — geometry is identical) ─
gdf_geom = gpd.read_file(GPKG_P2, layer='period_B_2018_2025')
gdf_geom['Mun_Code'] = gdf_geom['Mun_Code'].astype(str).str.zfill(5)
# Keep only geometry + Mun_Code from p2; all attributes come from p3_matrix
gdf_geom = gdf_geom[['Mun_Code', 'geometry']].copy()
print(f'Geometry loaded: {len(gdf_geom):,} municipalities, CRS: {gdf_geom.crs}')

In [ ]:
# ── Merge geometry with behavioural matrix ────────────────────────────────────
gdf = gdf_geom.merge(df_matrix, on='Mun_Code', how='left')

# Reproject to metric CRS for distance calculations
gdf = gdf.to_crs(CRS_METRIC)

n_missing = gdf['behavioural_group'].isna().sum()
print(f'Merged GDF: {len(gdf):,} municipalities | Missing behavioural_group: {n_missing}')

# ── Rural subset ──────────────────────────────────────────────────────────────
gdf_rural = gdf[gdf['tipo_goerlich'].isin(RURAL_TYPES)].copy().reset_index(drop=True)
print(f'Rural municipalities: {len(gdf_rural):,}')
print(gdf_rural.groupby(['tipo_goerlich', 'behavioural_group']).size().unstack(fill_value=0))

---
## 2 · Distance to nearest urban centre (peri-urban proxy)

In [ ]:
# ── Load p0 Goerlich GeoPackage to get urban and intermediate centroids ───────
import fiona
layers = fiona.listlayers(GPKG_P0_GOERLICH)
print(f'Layers in p0 Goerlich GeoPackage: {layers}')

In [ ]:
# ── Load p0 Goerlich GeoPackage (has tipo_goerlich + geometry) ───────────────
GOERLICH_LAYER = 'municipios_goerlich_2025'   # confirm with fiona output above

gdf_goerlich = gpd.read_file(GPKG_P0_GOERLICH, layer=GOERLICH_LAYER)
gdf_goerlich['Mun_Code'] = gdf_goerlich['Mun_Code'].astype(str).str.zfill(5)
gdf_goerlich = gdf_goerlich.to_crs(CRS_METRIC)
print(f'p0 Goerlich loaded: {len(gdf_goerlich):,} municipalities')
print(f'Columns: {list(gdf_goerlich.columns)}')

In [ ]:
# ── Extract urban and intermediate centroids ──────────────────────────────────
gdf_urban = gdf_goerlich[gdf_goerlich['tipo_goerlich'].isin(URBAN_TYPES)].copy()
gdf_urban['centroid'] = gdf_urban.geometry.centroid
urban_centroids = gdf_urban.set_geometry('centroid')[['Mun_Code', 'centroid']]
print(f'Urban municipalities (centroids): {len(urban_centroids):,}')

gdf_intermediate = gdf_goerlich[gdf_goerlich['tipo_goerlich'].isin(INTERMEDIATE_TYPES)].copy()
gdf_intermediate['centroid'] = gdf_intermediate.geometry.centroid
intermediate_centroids = gdf_intermediate.set_geometry('centroid')[['Mun_Code', 'centroid']]
print(f'Intermediate municipalities (centroids): {len(intermediate_centroids):,}')

In [ ]:
print(gdf_goerlich['tipo_goerlich'].value_counts())

In [ ]:
# ── Compute distance from each rural municipality to nearest urban centroid ───
gdf_rural_centroids = gdf_rural.copy()
gdf_rural_centroids['centroid'] = gdf_rural_centroids.geometry.centroid
gdf_rural_c = gdf_rural_centroids.set_geometry('centroid')[['Mun_Code', 'centroid']]

# ── Distance to nearest urban centre ─────────────────────────────────────────
nearest_urban = gpd.sjoin_nearest(
    gdf_rural_c,
    urban_centroids.rename(columns={'Mun_Code': 'nearest_urban_code'}),
    how='left',
    distance_col='dist_nearest_urban_m'
)
nearest_urban['dist_nearest_urban_km'] = nearest_urban['dist_nearest_urban_m'] / 1000
nearest_urban['within_60km_urban'] = nearest_urban['dist_nearest_urban_km'] <= PERIURBAN_KM

# ── Distance to nearest intermediate centre ───────────────────────────────────
nearest_intermediate = gpd.sjoin_nearest(
    gdf_rural_c,
    intermediate_centroids.rename(columns={'Mun_Code': 'nearest_intermediate_code'}),
    how='left',
    distance_col='dist_nearest_intermediate_m'
)
nearest_intermediate['dist_nearest_intermediate_km'] = nearest_intermediate['dist_nearest_intermediate_m'] / 1000
nearest_intermediate['within_60km_intermediate'] = nearest_intermediate['dist_nearest_intermediate_km'] <= PERIURBAN_KM

# ── Merge both distance sets back onto rural GDF ──────────────────────────────
dist_cols_urban = ['Mun_Code', 'nearest_urban_code', 'dist_nearest_urban_km', 'within_60km_urban']
dist_cols_inter = ['Mun_Code', 'nearest_intermediate_code', 'dist_nearest_intermediate_km', 'within_60km_intermediate']

gdf_rural = gdf_rural.merge(
    nearest_urban[dist_cols_urban].drop_duplicates('Mun_Code'), on='Mun_Code', how='left'
)
gdf_rural = gdf_rural.merge(
    nearest_intermediate[dist_cols_inter].drop_duplicates('Mun_Code'), on='Mun_Code', how='left'
)

# ── Categorical proximity column ──────────────────────────────────────────────
def assign_proximity(row):
    if row['within_60km_urban']:
        return 'near_urban'
    elif row['within_60km_intermediate']:
        return 'near_intermediate'
    else:
        return 'remote'

gdf_rural['urban_proximity'] = gdf_rural.apply(assign_proximity, axis=1)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'Distance computed for {gdf_rural["dist_nearest_urban_km"].notna().sum():,} rural municipalities')
print()
print('=== urban_proximity by typology ===')
print(gdf_rural.groupby(['tipo_goerlich', 'urban_proximity']).size().unstack(fill_value=0))

In [ ]:
print(gdf_rural[gdf_rural['tipo_goerlich'] == 'Rural - Remoto']
      .groupby(['urban_proximity', 'behavioural_group'])
      .size().unstack(fill_value=0))

In [ ]:
# ── Remote Rural-Remote municipalities with positive growth in Period B ────────
# Validates the narrative in the results section: 59 genuinely remote Rural-Remote
# municipalities, of which 18 show positive var_acum_pct_B > 0.

remote_positive = gdf_rural[
    (gdf_rural['urban_proximity'] == 'remote') &
    (gdf_rural['var_acum_pct_B'] > 0)
][['Mun_Code', 'Mun_Name', 'Comarca_Name', 'Prov_Name', 'CCAA_Name',
   'behavioural_group', 'var_acum_pct_A', 'var_acum_pct_B',
   'dist_nearest_urban_km', 'dist_nearest_intermediate_km']
].sort_values('var_acum_pct_B', ascending=False).reset_index(drop=True)

print(f'Remote Rural-Remote municipalities with positive growth in Period B: {len(remote_positive)}')
print()
print(remote_positive.to_string())

print()
print('=== Behavioural group breakdown ===')
print(remote_positive['behavioural_group'].value_counts())

print()
print('=== Geographic concentration ===')
print(remote_positive.groupby(['Prov_Name', 'Comarca_Name'])[['Mun_Name']].count().rename(
    columns={'Mun_Name': 'n_municipalities'}
).sort_values('n_municipalities', ascending=False))

---
## 3 · Pipeline A — Reverters (`Reverses in B`)

Primary neo-rural signal: municipalities that declined in Period A and grew in Period B.
Applied to both Rural-Remote and Rural-Accessible.

### 3.1 · Spatial outlier classification

In [ ]:
# ── Build Queen contiguity for ALL municipalities (full GDF) ──────────────────
# Built on the full 8,132-municipality GDF so that urban and intermediate
# neighbours are included when computing mean neighbour var_acum_pct_B
# for rural target municipalities.

def build_queen_neighbours(gdf, id_col='Mun_Code', buffer_m=1):
    """
    Returns a dict {mun_code: [list of neighbour codes]} using Queen contiguity
    (shared boundary or vertex). A small buffer closes topological gaps.
    buffer_m: buffer in metres (default 1m).
    """
    gdf_buffered = gdf[[id_col, 'geometry']].copy()
    gdf_buffered['geometry'] = gdf_buffered.geometry.buffer(buffer_m)
    touches = gpd.sjoin(gdf_buffered, gdf_buffered,
                        how='left', predicate='intersects',
                        lsuffix='left', rsuffix='right')
    touches = touches[touches[id_col + '_left'] != touches[id_col + '_right']]
    neighbours = (
        touches.groupby(id_col + '_left')[id_col + '_right']
        .apply(list).to_dict()
    )
    for code in gdf[id_col]:
        if code not in neighbours:
            neighbours[code] = []
    return neighbours

print('Building Queen contiguity for all 8,132 municipalities...')
queen_neighbours = build_queen_neighbours(gdf)
n_isolated = sum(1 for v in queen_neighbours.values() if len(v) == 0)
print(f'Queen contiguity built. Municipalities with no neighbours: {n_isolated}')
print(f'  (Note: island municipalities may appear isolated)')

# ── Diagnose remaining NaN cases after contiguity build ──────────────────────
print('\nDiagnosis — municipalities with 0 neighbours:')
for code, neighs in queen_neighbours.items():
    if len(neighs) == 0:
        row = gdf[gdf['Mun_Code'] == code][['Mun_Code', 'Mun_Name', 'Prov_Name']].values
        print(f'  {row}')

In [ ]:
# ── Compute mean and weighted-mean neighbour var_acum_pct_B ──────────────────
pop_b = gdf.set_index('Mun_Code')['pop_start_B']
var_b = gdf.set_index('Mun_Code')['var_acum_pct_B']
mean_neigh_B   = {}
wmean_neigh_B  = {}
n_neigh        = {}
for code, neighs in queen_neighbours.items():
    # Filter neighbours present in the full GDF (all typologies)
    valid = [n for n in neighs if n in var_b.index and not pd.isna(var_b[n])]
    n_neigh[code] = len(valid)
    if valid:
        vals  = var_b[valid].values
        pops  = pop_b[valid].values
        mean_neigh_B[code]  = np.mean(vals)
        wmean_neigh_B[code] = np.average(vals, weights=pops) if pops.sum() > 0 else np.mean(vals)
    else:
        mean_neigh_B[code]  = np.nan
        wmean_neigh_B[code] = np.nan

gdf_rural['n_queen_neighbours']    = gdf_rural['Mun_Code'].map(n_neigh)
gdf_rural['mean_neigh_var_B']      = gdf_rural['Mun_Code'].map(mean_neigh_B)
gdf_rural['wmean_neigh_var_B']     = gdf_rural['Mun_Code'].map(wmean_neigh_B)

print('Neighbour mean computed.')
print(f'  Mean neigh var_B — median: {gdf_rural["mean_neigh_var_B"].median():.2f}%')
print(f'  NaN (no rural neighbours): {gdf_rural["mean_neigh_var_B"].isna().sum()}')

# ── Diagnose remaining NaN ────────────────────────────────────────────────────
nan_codes = set(gdf_rural[gdf_rural['mean_neigh_var_B'].isna()]['Mun_Code'])
for code in nan_codes:
    neighs = queen_neighbours.get(code, [])
    valid  = [n for n in neighs if n in var_b.index and not pd.isna(var_b[n])]
    print(f'{code}: {len(neighs)} neighbours, {len(valid)} with valid var_acum_pct_B')

In [ ]:
print('Municipalities with NaN mean_neigh_var_B:')
print(gdf_rural[gdf_rural['mean_neigh_var_B'].isna()][['Mun_Code', 'Mun_Name', 'Prov_Name', 'tipo_goerlich', 'behavioural_group']].to_string())

In [ ]:
nan_codes = set(gdf_rural[gdf_rural['mean_neigh_var_B'].isna()]['Mun_Code'])
for code in nan_codes:
    neighs = queen_neighbours.get(code, [])
    print(f'{code}: {len(neighs)} neighbours in queen_neighbours')

In [ ]:
# ── Classify spatial outliers for Reverters ───────────────────────────────────
# Condition: behavioural_group == 'Reverses in B' AND mean_neigh_var_B < 0

mask_reverter = (
    (gdf_rural['behavioural_group'] == 'Reverses in B') &
    (gdf_rural['var_acum_pct_B'] > 0)
)
gdf_rural['is_reverter'] = mask_reverter
gdf_rural['reverter_spatial_outlier'] = (
    mask_reverter & (gdf_rural['mean_neigh_var_B'] < 0)
)

print('=== Reverter spatial outlier classification ===')
print()
for typol in RURAL_TYPES:
    sub = gdf_rural[gdf_rural['tipo_goerlich'] == typol]
    n_rev     = sub['is_reverter'].sum()
    n_outlier = sub['reverter_spatial_outlier'].sum()
    n_cluster = n_rev - n_outlier
    n_no_neigh = (sub['is_reverter'] & sub['mean_neigh_var_B'].isna()).sum()
    print(f'{typol}:')
    print(f'  Total reverters              : {n_rev:,}')
    print(f'  Spatial outliers (isolated)  : {n_outlier:,}  ({100*n_outlier/n_rev:.1f}%)')
    print(f'  Non-isolated (clusters/mixed): {n_cluster:,}  ({100*n_cluster/n_rev:.1f}%)')
    print(f'  No rural neighbours (islands): {n_no_neigh:,}')
    print()

In [ ]:
print('Reverters with var_acum_pct_B == 0:')
print(gdf_rural[
    (gdf_rural['behavioural_group'] == 'Reverses in B') &
    (gdf_rural['var_acum_pct_B'] == 0)
][['Mun_Code', 'Mun_Name', 'pop_start_B', 'pop_end_B']].to_string())

### 3.2 · Connected-component clusters (direct Queen contact)

In [ ]:
def find_direct_clusters(gdf_target, queen_neighbours, group_col='Mun_Code', prefix='rev'):
    """
    Finds connected components among target municipalities using Queen contiguity.
    Two target municipalities are connected if they share a Queen boundary.
    Returns a Series mapping Mun_Code -> cluster_id (NaN for isolated municipalities).
    prefix: string prefix for cluster IDs (e.g. 'rev' or 'dyn').
    """
    target_codes = set(gdf_target[group_col])
    G = nx.Graph()
    G.add_nodes_from(target_codes)
    for code in target_codes:
        for neigh in queen_neighbours.get(code, []):
            if neigh in target_codes:
                G.add_edge(code, neigh)

    cluster_map = {}
    cluster_id  = 1
    for component in nx.connected_components(G):
        if len(component) >= 2:
            cid = f'{prefix}_cluster_{cluster_id:03d}'
            for code in component:
                cluster_map[code] = cid
            cluster_id += 1
        else:
            # Single node: isolated
            (code,) = component
            cluster_map[code] = f'{prefix}_isolated'

    return cluster_map


# Apply to reverters
gdf_reverters_all = gdf_rural[gdf_rural['is_reverter']].copy()
rev_direct_clusters = find_direct_clusters(gdf_reverters_all, queen_neighbours, prefix='rev')
gdf_rural['rev_direct_cluster'] = gdf_rural['Mun_Code'].map(rev_direct_clusters)

# Summary
cluster_counts = (
    pd.Series(rev_direct_clusters)
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'cluster_id', 0: 'n_municipalities'})
)
n_clusters = (cluster_counts['cluster_id'] != 'rev_isolated').sum()
n_in_clusters = cluster_counts[cluster_counts['cluster_id'] != 'rev_isolated']['count'].sum() \
    if 'count' in cluster_counts.columns \
    else cluster_counts[cluster_counts['cluster_id'] != 'rev_isolated'].iloc[:, 1].sum()

print(f'=== Reverter direct clusters (Queen contiguity) ===')
print(f'Number of multi-municipality clusters: {n_clusters}')
print(f'Municipalities in clusters: {n_in_clusters}')
print()
print('Top 15 clusters by size:')
print(cluster_counts[cluster_counts.iloc[:, 0] != 'rev_isolated'].head(15).to_string(index=False))

In [ ]:
sizes = cluster_counts[cluster_counts.iloc[:, 0] != 'rev_isolated'].iloc[:, 1]
print('=== Cluster size distribution ===')
print(f'Clusters of 2-5 municipalities  : {(sizes.between(2,5)).sum()}')
print(f'Clusters of 6-20 municipalities : {(sizes.between(6,20)).sum()}')
print(f'Clusters of 21-50 municipalities: {(sizes.between(21,50)).sum()}')
print(f'Clusters of >50 municipalities  : {(sizes > 50).sum()}')
print(f'Isolated reverters              : {(pd.Series(rev_direct_clusters) == "rev_isolated").sum()}')

In [ ]:
# Cross large clusters with urban_proximity
large_cluster_ids = cluster_counts[cluster_counts.iloc[:,1] > 50].iloc[:,0].tolist()
large = gdf_rural[gdf_rural['rev_direct_cluster'].isin(large_cluster_ids)]
print('\n=== Large clusters (>50) × urban_proximity ===')
print(large.groupby(['rev_direct_cluster', 'urban_proximity']).size().unstack(fill_value=0))

In [ ]:
print(gdf_rural[
    (gdf_rural['rev_direct_cluster'] == 'rev_cluster_025') & 
    (gdf_rural['urban_proximity'] == 'remote')
][['Mun_Code', 'Mun_Name', 'Prov_Name', 'CCAA_Name', 'tipo_goerlich', 'var_acum_pct_B']].to_string())

In [ ]:
print(gdf_rural[gdf_rural['Mun_Code'] == '19310'][
    ['Mun_Code', 'Mun_Name', 'pop_start_B', 'pop_end_B', 'var_acum_pct_B']
].to_string())

In [ ]:
for cluster_id in ['rev_cluster_001', 'rev_cluster_002', 'rev_cluster_019', 'rev_cluster_066']:
    print(f'\n=== {cluster_id} ===')
    sub = gdf_rural[gdf_rural['rev_direct_cluster'] == cluster_id]
    print(f'n = {len(sub)}')
    print('\nCCAA:')
    print(sub.groupby('CCAA_Name').size().sort_values(ascending=False))
    print('\nProvincia:')
    print(sub.groupby('Prov_Name').size().sort_values(ascending=False))
    print('\nComarca (top 10):')
    print(sub.groupby('Comarca_Name').size().sort_values(ascending=False).head(10))
    print('\nurban_proximity:')
    print(sub.groupby('urban_proximity').size())

In [ ]:
print(gdf_rural.columns.tolist())
print()
print(gdf_rural['rev_direct_cluster'].value_counts().head(5))

In [ ]:
# Identificar clusters grandes por geografía, no por ID
cluster_geo = (
    gdf_rural[
        gdf_rural['rev_direct_cluster'].fillna('').str.startswith('rev_cluster')
    ]
    .groupby('rev_direct_cluster')
    .agg(
        n=('Mun_Code', 'count'),
        ccaa=('CCAA_Name', lambda x: ', '.join(sorted(x.unique()))),
        provincias=('Prov_Name', lambda x: ', '.join(sorted(x.unique()))),
        comarcas_top5=('Comarca_Name', lambda x: ', '.join(
            x.value_counts().head(5).index.tolist()
        )),
        near_urban=('urban_proximity', lambda x: (x == 'near_urban').sum()),
        near_intermediate=('urban_proximity', lambda x: (x == 'near_intermediate').sum()),
        remote=('urban_proximity', lambda x: (x == 'remote').sum()),
    )
    .sort_values('n', ascending=False)
    .head(10)
    .reset_index()
)
print(cluster_geo.to_string())

### 3.3 · Extended neighbourhood clusters (shared-neighbour proximity)

Two reverters that do not share a Queen boundary but share at least one
common Queen neighbour are considered spatially proximate. This captures
groupings where growing municipalities are near each other but separated
by one declining municipality.

In [ ]:
def find_extended_clusters(gdf_target, queen_neighbours, group_col='Mun_Code', prefix='rev'):
    """
    Extended clusters: two target municipalities are linked if they share
    at least one common Queen neighbour (even if they don't touch directly).
    Returns a Series mapping Mun_Code -> extended_cluster_id.
    """
    target_codes = set(gdf_target[group_col])
    G = nx.Graph()
    G.add_nodes_from(target_codes)
    # Direct edges (same as direct clusters)
    for code in target_codes:
        for neigh in queen_neighbours.get(code, []):
            if neigh in target_codes:
                G.add_edge(code, neigh)
    # Extended edges: shared neighbours
    target_list = list(target_codes)
    for i, code_a in enumerate(target_list):
        neighs_a = set(queen_neighbours.get(code_a, []))
        for code_b in target_list[i+1:]:
            if G.has_edge(code_a, code_b):
                continue   # already directly connected
            neighs_b = set(queen_neighbours.get(code_b, []))
            if neighs_a & neighs_b:  # non-empty intersection
                G.add_edge(code_a, code_b)
    cluster_map = {}
    cluster_id  = 1
    for component in nx.connected_components(G):
        if len(component) >= 2:
            cid = f'{prefix}_ext_cluster_{cluster_id:03d}'
            for code in component:
                cluster_map[code] = cid
            cluster_id += 1
        else:
            (code,) = component
            cluster_map[code] = f'{prefix}_ext_isolated'
    return cluster_map

# Ensure gdf_reverters_all reflects current is_reverter mask (var_acum_pct_B > 0)
gdf_reverters_all = gdf_rural[gdf_rural['is_reverter']].copy()

print('Computing extended neighbourhood clusters for reverters...')
rev_extended_clusters = find_extended_clusters(gdf_reverters_all, queen_neighbours, prefix='rev')
gdf_rural['rev_extended_cluster'] = gdf_rural['Mun_Code'].map(rev_extended_clusters)

# Comparison: how many isolated in direct become part of extended cluster?
direct_isolated = {k for k, v in rev_direct_clusters.items() if v == 'rev_isolated'}
extended_joined = {k for k in direct_isolated if rev_extended_clusters.get(k, '').startswith('rev_ext_cluster')}
print(f'Reverters isolated in direct clusters: {len(direct_isolated):,}')
print(f'Of those, pulled into extended clusters: {len(extended_joined):,}')

ext_cluster_counts = (
    pd.Series(rev_extended_clusters)
    .value_counts()
    .reset_index()
)
n_ext_clusters = (ext_cluster_counts.iloc[:, 0].str.startswith('rev_ext_cluster')).sum()
print(f'Extended multi-municipality clusters: {n_ext_clusters}')
print()
print('Top 15 extended clusters by size:')
mask_clusters = ext_cluster_counts.iloc[:, 0].str.startswith('rev_ext_cluster')
print(ext_cluster_counts[mask_clusters].head(15).to_string(index=False))

### 3.4 · LISA — Local Moran's I (statistical validation)
- LISA applied to ALL municipalities (n ≈ 8,100)
- W built on full contiguity graph; results extracted for rural stratum only.
- This avoids the ghost-neighbour problem: rural municipalities bordering
urban/intermediate areas retain their true topological context.
- Significance threshold: p < 0.05 (conditional permutation, 9,999 iterations).

In [ ]:
# ── LISA on full municipal GDF (all typologies, n ≈ 8,100) ───────────────────
# RATIONALE: W must reflect the true topological neighbourhood of each
# municipality. Building W on the rural subset only creates ghost neighbours:
# rural municipalities bordering urban/intermediate areas appear artificially
# isolated, inflating ns counts at rural-urban boundaries.
# Solution: build W on the full GDF; extract results for rural stratum only.
# Permutations: 9,999 → p_min = 0.0001, stable randomisation distribution.
# Significance threshold: p < 0.05 (Recaño Valverde & Marbán, 2024).

if LISA_AVAILABLE:

    # ── Build full-GDF spatial weights matrix ─────────────────────────────────
    gdf_all_lisa = gdf.reset_index(drop=True).copy()
    gdf_all_lisa['var_acum_pct_B_filled'] = gdf_all_lisa['var_acum_pct_B'].fillna(0)

    print('Building Queen spatial weights matrix on full GDF (~8,100 municipalities)...')
    w = QueenW.from_dataframe(gdf_all_lisa, use_index=False)
    w.transform = 'R'   # row-standardise

    y = gdf_all_lisa['var_acum_pct_B_filled'].values

    # ── Global Moran's I ──────────────────────────────────────────────────────
    moran_global = Moran(y, w, permutations=LISA_PERMUT)
    print(f"\nGlobal Moran's I (all municipalities, var_acum_pct_B):")
    print(f'  I = {moran_global.I:.4f}')
    print(f'  p = {moran_global.p_sim:.4f}')
    print(f'  z = {moran_global.z_sim:.4f}')

    # ── Local Moran's I ───────────────────────────────────────────────────────
    print('\nRunning Local Moran\'s I (this may take 2–4 min with 9,999 permutations)...')
    moran_local = Moran_Local(y, w, permutations=LISA_PERMUT, seed=42)

    quadrant_labels = {1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}

    gdf_all_lisa['lisa_I']           = moran_local.Is
    gdf_all_lisa['lisa_p_sim']       = moran_local.p_sim
    gdf_all_lisa['lisa_quadrant']    = moran_local.q
    gdf_all_lisa['lisa_quad_label']  = gdf_all_lisa['lisa_quadrant'].map(quadrant_labels)
    gdf_all_lisa['lisa_significant'] = gdf_all_lisa['lisa_p_sim'] < LISA_ALPHA
    gdf_all_lisa['lisa_sig_quad']    = np.where(
        gdf_all_lisa['lisa_significant'],
        gdf_all_lisa['lisa_quad_label'], 'ns'
    )

    # ── Extract results for rural stratum only ────────────────────────────────
    lisa_rural = (
        gdf_all_lisa[gdf_all_lisa['tipo_goerlich'].isin(RURAL_TYPES)]
        [['Mun_Code', 'lisa_I', 'lisa_p_sim', 'lisa_sig_quad']]
        .set_index('Mun_Code')
        .to_dict('index')
    )

    gdf_rural['lisa_I']        = gdf_rural['Mun_Code'].map(lambda x: lisa_rural.get(x, {}).get('lisa_I'))
    gdf_rural['lisa_p_sim']    = gdf_rural['Mun_Code'].map(lambda x: lisa_rural.get(x, {}).get('lisa_p_sim'))
    gdf_rural['lisa_sig_quad'] = gdf_rural['Mun_Code'].map(lambda x: lisa_rural.get(x, {}).get('lisa_sig_quad'))

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f'\nLISA quadrant counts — all municipalities (significant only, p < {LISA_ALPHA}):')
    print(gdf_all_lisa[gdf_all_lisa['lisa_significant']]['lisa_sig_quad'].value_counts())

    print('\nLISA quadrant counts — rural stratum by typology (significant only):')
    print(gdf_rural[gdf_rural['lisa_sig_quad'] != 'ns']
          .groupby(['tipo_goerlich', 'lisa_sig_quad']).size().unstack(fill_value=0))

    print(f'\nRural ns before correction (rural-only W): [run old code to compare]')
    print(f'Rural significant (full W): {(gdf_rural["lisa_sig_quad"] != "ns").sum():,}')
    print(f'Rural ns (full W):          {(gdf_rural["lisa_sig_quad"] == "ns").sum():,}')

else:
    print('LISA skipped — esda/libpysal not available.')
    print('Install with: pip install esda libpysal')
    gdf_rural['lisa_I']        = np.nan
    gdf_rural['lisa_p_sim']    = np.nan
    gdf_rural['lisa_sig_quad'] = np.nan

### 3.5 · Reverter summary table

In [ ]:
# ── Consolidated reverter classification ─────────────────────────────────────
# Each reverter gets: spatial_outlier flag, direct_cluster_id,
# extended_cluster_id, within_60km_urban, lisa_sig_quad

df_rev = gdf_rural[gdf_rural['is_reverter']].copy()

# Assign a final spatial class:
#   'spatial_outlier'   : grows, mean_neigh_var_B < 0, no growing direct neighbours
#   'direct_cluster'    : member of a direct contiguity cluster
#   'extended_cluster'  : pulled into cluster only via shared neighbour
#   'ambiguous'         : grows, mean_neigh_var_B >= 0 (growing neighbourhood)

def assign_spatial_class(row):
    if pd.isna(row['rev_direct_cluster']):
        return 'not_reverter'
    if row['rev_direct_cluster'].startswith('rev_cluster'):
        return 'direct_cluster'
    if row['rev_extended_cluster'].startswith('rev_ext_cluster'):
        return 'extended_cluster'
    # Isolated in both — check outlier criterion
    if row['reverter_spatial_outlier']:
        return 'spatial_outlier'
    return 'ambiguous'

df_rev['spatial_class'] = df_rev.apply(assign_spatial_class, axis=1)

print('=== Reverter spatial classification ===')
print()
print(df_rev.groupby(['tipo_goerlich', 'spatial_class']).size().unstack(fill_value=0))
print()
print('=== Reverter spatial class × peri-urban flag ===')
print(df_rev.groupby(['spatial_class', 'within_60km_urban']).size().unstack(fill_value=0))

In [ ]:
# ── Display top reverter municipalities ──────────────────────────────────────
display_cols = [
    'Mun_Code', 'Mun_Name', 'Prov_Name', 'CCAA_Name', 'tipo_goerlich',
    'Pop_ref', 'var_acum_pct_A', 'var_acum_pct_B',
    'mean_neigh_var_B', 'dist_nearest_urban_km', 'within_60km_urban',
    'spatial_class', 'rev_direct_cluster', 'rev_extended_cluster', 'lisa_sig_quad'
]

print('=== Top 30 reverters by var_acum_pct_B — Rural Remoto ===')
top_rev_remote = (
    df_rev[df_rev['tipo_goerlich'] == 'Rural - Remoto']
    [display_cols]
    .sort_values('var_acum_pct_B', ascending=False)
    .head(30)
)
display(top_rev_remote)

print('\n=== Top 30 reverters by var_acum_pct_B — Rural Accesible ===')
top_rev_access = (
    df_rev[df_rev['tipo_goerlich'] == 'Rural - Accesible']
    [display_cols]
    .sort_values('var_acum_pct_B', ascending=False)
    .head(30)
)
display(top_rev_access)

In [ ]:
print(gdf_rural[gdf_rural['rev_direct_cluster'] == 'rev_cluster_035']
      [['Mun_Name', 'Prov_Name', 'CCAA_Name']].value_counts('CCAA_Name'))

---
## 4 · Pipeline B — Sustained dynamisers (`Grows in both`)

Structurally resilient municipalities that maintained positive demographic growth
across both periods. These include cases like Negueira de Muñiz (Lugo) that never
entered decline. Conceptually distinct from reverters but spatially they may form
the core of the same clusters — this is tested explicitly here.

### 4.1 · Spatial outlier classification

In [ ]:
mask_dynamiser = (
    (gdf_rural['behavioural_group'] == 'Grows in both') &
    (gdf_rural['var_acum_pct_A'] > 0) &
    (gdf_rural['var_acum_pct_B'] > 0)
)

gdf_rural['is_dynamiser'] = mask_dynamiser
gdf_rural['dynamiser_spatial_outlier'] = (
    mask_dynamiser & (gdf_rural['mean_neigh_var_B'] < 0)
)

print('=== Dynamiser spatial outlier classification ===')
print()
for typol in RURAL_TYPES:
    sub = gdf_rural[gdf_rural['tipo_goerlich'] == typol]
    n_dyn     = sub['is_dynamiser'].sum()
    n_outlier = sub['dynamiser_spatial_outlier'].sum()
    n_cluster = n_dyn - n_outlier
    n_no_neigh = (sub['is_dynamiser'] & sub['mean_neigh_var_B'].isna()).sum()
    print(f'{typol}:')
    print(f'  Total dynamisers             : {n_dyn:,}')
    pct = (100*n_outlier/n_dyn) if n_dyn > 0 else 0
    print(f'  Spatial outliers (isolated)  : {n_outlier:,}  ({pct:.1f}%)')
    print(f'  Non-isolated (clusters/mixed): {n_cluster:,}')
    print(f'  No neighbours (islands)      : {n_no_neigh:,}')
    print()

### 4.2 · Connected-component clusters (direct Queen contact)

In [ ]:
gdf_dynamisers_all = gdf_rural[gdf_rural['is_dynamiser']].copy()
dyn_direct_clusters = find_direct_clusters(gdf_dynamisers_all, queen_neighbours, prefix='dyn')
gdf_rural['dyn_direct_cluster'] = gdf_rural['Mun_Code'].map(dyn_direct_clusters)

dyn_cluster_counts = pd.Series(dyn_direct_clusters).value_counts().reset_index()
n_dyn_clusters = (dyn_cluster_counts.iloc[:, 0].str.startswith('dyn_cluster')).sum()
print(f'Dynamiser direct clusters: {n_dyn_clusters}')
print()
print('Top 15 dynamiser clusters by size:')
mask_dyn = dyn_cluster_counts.iloc[:, 0].str.startswith('dyn_cluster')
print(dyn_cluster_counts[mask_dyn].head(15).to_string(index=False))

In [ ]:
cluster_dyn_geo = (
    gdf_rural[
        gdf_rural['dyn_direct_cluster'].fillna('').str.startswith('dyn_cluster')
    ]
    .groupby('dyn_direct_cluster')
    .agg(
        n=('Mun_Code', 'count'),
        ccaa=('CCAA_Name', lambda x: ', '.join(sorted(x.unique()))),
        provincias=('Prov_Name', lambda x: ', '.join(sorted(x.unique()))),
        comarcas_top5=('Comarca_Name', lambda x: ', '.join(
            x.value_counts().head(5).index.tolist()
        )),
        near_urban=('urban_proximity', lambda x: (x == 'near_urban').sum()),
        near_intermediate=('urban_proximity', lambda x: (x == 'near_intermediate').sum()),
        remote=('urban_proximity', lambda x: (x == 'remote').sum()),
        mean_var_A=('var_acum_pct_A', 'mean'),
        mean_var_B=('var_acum_pct_B', 'mean'),
    )
    .sort_values('n', ascending=False)
    .head(10)
    .reset_index()
)
print(cluster_dyn_geo.to_string())

### 4.3 · Extended neighbourhood clusters

In [ ]:
print('Computing extended neighbourhood clusters for dynamisers...')
dyn_extended_clusters = find_extended_clusters(gdf_dynamisers_all, queen_neighbours, prefix='dyn')
gdf_rural['dyn_extended_cluster'] = gdf_rural['Mun_Code'].map(dyn_extended_clusters)

dyn_direct_iso   = {k for k, v in dyn_direct_clusters.items() if v == 'dyn_isolated'}
dyn_ext_joined   = {k for k in dyn_direct_iso if dyn_extended_clusters.get(k, '').startswith('dyn_ext_cluster')}
print(f'Dynamisers isolated in direct clusters: {len(dyn_direct_iso):,}')
print(f'Of those, pulled into extended clusters: {len(dyn_ext_joined):,}')

dyn_ext_counts = pd.Series(dyn_extended_clusters).value_counts().reset_index()
n_dyn_ext = (dyn_ext_counts.iloc[:, 0].str.startswith('dyn_ext_cluster')).sum()
print(f'Extended dynamiser clusters: {n_dyn_ext}')

### 4.4 · LISA validation for dynamisers

LISA was run on the full rural stratum in Section 3.4. Here we cross-reference
dynamiser municipalities with the LISA results.

In [ ]:
df_dyn = gdf_rural[gdf_rural['is_dynamiser']].copy()

if 'lisa_sig_quad' in df_dyn.columns:
    print('=== LISA quadrant distribution for sustained dynamisers ===')
    print(df_dyn.groupby(['tipo_goerlich', 'lisa_sig_quad']).size().unstack(fill_value=0))
else:
    print('LISA not available — skipping cross-reference.')

### 4.5 · Dynamiser summary table

In [ ]:
def assign_dyn_spatial_class(row):
    if pd.isna(row['dyn_direct_cluster']):
        return 'not_dynamiser'
    if row['dyn_direct_cluster'].startswith('dyn_cluster'):
        return 'direct_cluster'
    if row['dyn_extended_cluster'].startswith('dyn_ext_cluster'):
        return 'extended_cluster'
    if row['dynamiser_spatial_outlier']:
        return 'spatial_outlier'
    return 'ambiguous'

df_dyn['spatial_class'] = df_dyn.apply(assign_dyn_spatial_class, axis=1)

print('=== Dynamiser spatial classification ===')
print(df_dyn.groupby(['tipo_goerlich', 'spatial_class']).size().unstack(fill_value=0))

print('\n=== Top 30 dynamisers by var_acum_pct_B — Rural Remoto ===')
display_cols_dyn = [
    'Mun_Code', 'Mun_Name', 'Prov_Name', 'CCAA_Name', 'tipo_goerlich',
    'Pop_ref', 'var_acum_pct_A', 'var_acum_pct_B',
    'mean_neigh_var_B', 'dist_nearest_urban_km', 'within_60km_urban',
    'spatial_class', 'dyn_direct_cluster', 'dyn_extended_cluster', 'lisa_sig_quad'
]
top_dyn_remote = (
    df_dyn[df_dyn['tipo_goerlich'] == 'Rural - Remoto']
    [display_cols_dyn]
    .sort_values('var_acum_pct_B', ascending=False)
    .head(30)
)
display(top_dyn_remote)

print('\n=== Top 30 dynamisers by var_acum_pct_B — Rural Accesible ===')
top_dyn_access = (
    df_dyn[df_dyn['tipo_goerlich'] == 'Rural - Accesible']
    [display_cols_dyn]
    .sort_values('var_acum_pct_B', ascending=False)
    .head(30)
)
display(top_dyn_access)

---
## 5 · Outputs — GeoPackage layers and CSV tables
| `lisa_results_all`   | LISA quadrant and p-value for all ~8,100 municipalities |

| `lisa_results_rural` | LISA quadrant and p-value for rural stratum only        |

In [ ]:
# ── Finalise attribute columns on gdf_rural before export ────────────────────
# Merge spatial_class from both pipelines back into main rural GDF

gdf_rural = gdf_rural.merge(
    df_rev[['Mun_Code', 'spatial_class']].rename(columns={'spatial_class': 'rev_spatial_class'}),
    on='Mun_Code', how='left'
)
gdf_rural = gdf_rural.merge(
    df_dyn[['Mun_Code', 'spatial_class']].rename(columns={'spatial_class': 'dyn_spatial_class'}),
    on='Mun_Code', how='left'
)

In [ ]:
# ── Export GeoPackage ─────────────────────────────────────────────────────────
OUT_GPKG = SPATIAL_PROC / 'p4_spatial_hotspots.gpkg'

# Layer 1: Rural-Remote all
gdf_rural[gdf_rural['tipo_goerlich'] == 'Rural - Remoto'].to_file(
    OUT_GPKG, layer='rural_remote_all', driver='GPKG'
)
print('Layer written: rural_remote_all')

# Layer 2: Rural-Accessible all
gdf_rural[gdf_rural['tipo_goerlich'] == 'Rural - Accesible'].to_file(
    OUT_GPKG, layer='rural_accessible_all', driver='GPKG'
)
print('Layer written: rural_accessible_all')

# Layer 3: Reverters classified
df_rev_geo = gdf_rural[gdf_rural['is_reverter']].copy()
df_rev_geo.to_file(OUT_GPKG, layer='reverters_classified', driver='GPKG')
print('Layer written: reverters_classified')

# Layer 4: Dynamisers classified
df_dyn_geo = gdf_rural[gdf_rural['is_dynamiser']].copy()
df_dyn_geo.to_file(OUT_GPKG, layer='dynamisers_classified', driver='GPKG')
print('Layer written: dynamisers_classified')

# Layer 5: LISA results — full municipal GDF (all typologies)
# Exported for cartographic use in QGIS; rural focus applied at map level.
gdf_all_lisa.drop(columns=['var_acum_pct_B_filled']).to_file(
    OUT_GPKG, layer='lisa_results_all', driver='GPKG'
)
print('Layer written: lisa_results_all')

# Layer 6: LISA results — rural stratum only (Rural Remoto + Rural Accesible)
gdf_rural.to_file(OUT_GPKG, layer='lisa_results_rural', driver='GPKG')
print('Layer written: lisa_results_rural')

print(f'\nGeoPackage exported: {OUT_GPKG.name}')

In [ ]:
# ── Export CSV tables ─────────────────────────────────────────────────────────
export_cols_rev = [
    'Mun_Code', 'Mun_Name', 'Comarca_Name', 'Prov_Name', 'CCAA_Name',
    'tipo_goerlich', 'size_group', 'Pop_ref',
    'var_acum_pct_A', 'var_anual_media_pct_A',
    'var_acum_pct_B', 'var_anual_media_pct_B',
    'behavioural_group',
    'n_queen_neighbours', 'mean_neigh_var_B', 'wmean_neigh_var_B',
    'dist_nearest_urban_km', 'nearest_urban_code', 'within_60km_urban',
    'urban_proximity',                                                   # ← añadido
    'reverter_spatial_outlier',
    'rev_direct_cluster', 'rev_extended_cluster',
    'rev_spatial_class',
    'lisa_sig_quad', 'lisa_I', 'lisa_p_sim'
]

export_cols_dyn = [
    'Mun_Code', 'Mun_Name', 'Comarca_Name', 'Prov_Name', 'CCAA_Name',
    'tipo_goerlich', 'size_group', 'Pop_ref',
    'var_acum_pct_A', 'var_anual_media_pct_A',
    'var_acum_pct_B', 'var_anual_media_pct_B',
    'behavioural_group',
    'n_queen_neighbours', 'mean_neigh_var_B', 'wmean_neigh_var_B',
    'dist_nearest_urban_km', 'nearest_urban_code', 'within_60km_urban',
    'urban_proximity',                                                   # ← añadido
    'dynamiser_spatial_outlier',
    'dyn_direct_cluster', 'dyn_extended_cluster',
    'dyn_spatial_class',
    'lisa_sig_quad', 'lisa_I', 'lisa_p_sim'
]

# Filter to available columns
rev_avail = [c for c in export_cols_rev if c in gdf_rural.columns]
dyn_avail = [c for c in export_cols_dyn if c in gdf_rural.columns]

out_rev = DEMO_DERIV / 'p4_reverters_classified.csv'
out_dyn = DEMO_DERIV / 'p4_dynamisers_classified.csv'

gdf_rural[gdf_rural['is_reverter']][rev_avail].to_csv(
    out_rev, sep=';', encoding='utf-8-sig', index=False
)
print(f'Exported: {out_rev.name}  ({gdf_rural["is_reverter"].sum():,} rows)')

gdf_rural[gdf_rural['is_dynamiser']][dyn_avail].to_csv(
    out_dyn, sep=';', encoding='utf-8-sig', index=False
)
print(f'Exported: {out_dyn.name}  ({gdf_rural["is_dynamiser"].sum():,} rows)')

In [ ]:
# ── Cluster summary table ─────────────────────────────────────────────────────
# For each non-trivial cluster (direct or extended, both pipelines):
# cluster_id, n_municipalities, typologies present, CCAA, mean var_acum_pct_B,
# median population, n within 60km urban, n LISA HH

def cluster_summary(gdf_sub, cluster_col, pipeline_label):
    rows = []
    for cid, group in gdf_sub[gdf_sub[cluster_col].notna()].groupby(cluster_col):
        if 'isolated' in str(cid):
            continue
        rows.append({
            'pipeline'          : pipeline_label,
            'cluster_id'        : cid,
            'n_municipalities'  : len(group),
            'typologies'        : ', '.join(sorted(group['tipo_goerlich'].unique())),
            'ccaa'              : ', '.join(sorted(group['CCAA_Name'].unique())),
            'mean_var_acum_B'   : round(group['var_acum_pct_B'].mean(), 2),
            'median_pop_ref'    : group['Pop_ref'].median(),
            'n_within_60km'     : group['within_60km_urban'].sum(),
            'n_lisa_HH'         : (group.get('lisa_sig_quad', pd.Series(dtype=str)) == 'HH').sum(),
        })
    return pd.DataFrame(rows).sort_values('n_municipalities', ascending=False)

df_rev_full = gdf_rural[gdf_rural['is_reverter']]
df_dyn_full = gdf_rural[gdf_rural['is_dynamiser']]

summary_parts = []
for pipeline, df_sub, col in [
    ('reverter_direct',   df_rev_full, 'rev_direct_cluster'),
    ('reverter_extended', df_rev_full, 'rev_extended_cluster'),
    ('dynamiser_direct',  df_dyn_full, 'dyn_direct_cluster'),
    ('dynamiser_extended',df_dyn_full, 'dyn_extended_cluster'),
]:
    if col in df_sub.columns:
        summary_parts.append(cluster_summary(df_sub, col, pipeline))

df_cluster_summary = pd.concat(summary_parts, ignore_index=True)

out_clust = DEMO_DERIV / 'p4_cluster_summary.csv'
df_cluster_summary.to_csv(out_clust, sep=';', encoding='utf-8-sig', index=False)
print(f'Exported: {out_clust.name}  ({len(df_cluster_summary):,} clusters total)')
print()
print('=== Top 20 clusters by size (all pipelines) ===')
display(df_cluster_summary.head(20))

In [ ]:
print(gdf_rural[gdf_rural['is_dynamiser']]
      .groupby(['tipo_goerlich', 'urban_proximity']).size().unstack(fill_value=0))

In [ ]:
print(gdf_rural[
    (gdf_rural['urban_proximity'] == 'remote') &
    (gdf_rural['var_acum_pct_B'] > 0)
][['Mun_Code', 'Mun_Name', 'Prov_Name', 'CCAA_Name', 'tipo_goerlich', 
   'behavioural_group', 'var_acum_pct_B', 'Pop_ref']].to_string())

## 6 · Interpretation notes

---

### Methodological notes

**NaN cases in neighbour mean (n=5):**
Five rural municipalities return NaN for mean neighbour cumulative growth because
they have no valid contiguous neighbours in the dataset: Formentera (Illes Balears),
Llívia (Girona), and A Illa de Arousa (Pontevedra) are islands or exclaves with no
shared terrestrial boundary; Hacinas and Monasterio de la Sierra (Burgos) are
surrounded by non-municipalised communal territories (fazerías) with no registered
population. These five municipalities are excluded from the spatial outlier analysis
but retained in the GeoPackage with NaN in spatial classification columns.

**Zero-growth reverters excluded (n=83):**
83 municipalities classified as Reverses in B in p3 show exactly zero cumulative
growth in Period B (pop_start_B == pop_end_B). These are excluded from p4 analysis
by applying the stricter criterion var_acum_pct_B > 0. Stable population over 7 years
is not demographic growth; including these cases would inflate reverter counts
without adding analytical signal. Note: p3 uses >= 0 (behavioural classification),
p4 uses > 0 (spatial analysis) — this discrepancy is intentional and documented.

**Cluster ID non-determinism:**
Connected-component cluster IDs are assigned by networkx in non-deterministic
order across kernel restarts. Clusters should be identified by their municipal
composition, not by their numeric ID. The cluster_summary CSV provides CCAA
and typology composition for each cluster as stable identifiers.

**Peri-urban proxy limitation:**
The urban proximity variable uses Euclidean distance to the nearest urban or
intermediate centroid (threshold: 60 km). This is an acknowledged simplification —
road-network travel time would improve precision, particularly for mountain
municipalities that are close in straight-line distance but distant by road
(e.g. Pyrenean municipalities). Future refinement via OSRM is noted
(Goerlich correspondence, March 2026). The Goerlich (2016) typology already
incorporates accessibility by construction, so the Euclidean proxy serves as a
supplementary filter within Rural-Remote, not as a replacement for the typology.

**libpysal island warnings:**
The LISA section produces ~7 island warnings from libpysal's internal Queen
weight matrix (Formentera, Hacinas, Monasterio de la Sierra, Llívia, A Illa de
Arousa, Ceuta, Melilla), which does not apply the 1-metre topological buffer used
in build_queen_neighbours. These warnings do not affect LISA results — they reflect
minor topological gaps in the source geometries that are resolved in the custom
contiguity function but not in libpysal's internal computation.

**LISA spatial weights — full GDF rationale:**
The spatial weights matrix W is constructed on the full ~8,100-municipality GDF
(all typologies), not on the rural subset only. Restricting W to rural municipalities
would create ghost neighbours: rural municipalities bordering urban or intermediate
areas would appear artificially isolated, inflating non-significant counts at
rural-urban boundaries. W is built on the full contiguity graph; LISA results are
then extracted for the rural stratum only. Permutations: 9,999 (p_min = 0.0001).

---

### Section 2 — Proximity breakdown

**Urban proximity distribution:**
The urban proximity cross-tabulation reveals a fundamental asymmetry between
the two rural typologies. Rural-Accessible is almost entirely within 60 km of
an urban centre (3,835 of 3,885 municipalities, 98.7%), with only 50 municipalities
(1.3%) within intermediate proximity range, confirming that accessibility to urban
areas is effectively built into the Goerlich typology. Rural-Remote shows more
heterogeneity: 1,188 municipalities (41.9%) are near urban, 1,591 (56.0%) are
near intermediate, and 59 (2.1%) are genuinely remote — beyond 60 km of any
urban or intermediate centre.

**Remote Rural-Remote × behavioural group:**
Of the 59 genuinely remote Rural-Remote municipalities, 18 show positive
demographic growth in Period B (16 classified as Reverses in B; 2 classified
as Grows in both: Hombrados and Torrecuadrada de Molina, Guadalajara). These
represent the analytically purest neo-rural signal: their growth cannot be
attributed to peri-urbanisation or labour-market spillovers from nearby centres
by construction. They are priority candidates for qualitative fieldwork, as their
demographic turnaround must be explained by endogenous or amenity-driven factors.

Geographically these municipalities are concentrated in three distinct areas:
(1) the Serranía Celtibérica interior across the comarcas of Molina de Aragón,
Alcarria Alta, and Alcarria Baja in Guadalajara, and Cuenca del Jiloca in Teruel
(11 municipalities: Abánades, Ablanque, Armallones, Hombrados, Morenilla,
Riba de Saelices, Sacecorbo, Tierzo, Torrecuadrada de Molina, Anquela del
Pedregal, and Blancas); (2) the Val d'Aran and surrounding Pyrenean
municipalities in Lleida (Les, Bossòst, Canejan, Vilamòs — 4 municipalities);
and (3) three municipalities on El Hierro island (Frontera, Valverde, El Pinar
de El Hierro — Canarias), which are geographically remote but represent a
distinct insular dynamic not comparable to mainland rural depopulation.

No genuinely remote Rural-Remote dynamiser exists in the dataset beyond
Hombrados and Torrecuadrada de Molina (Guadalajara). All remaining
Rural-Remote dynamisers are either near urban (47) or near intermediate (78).
This confirms that structurally sustained demographic growth in Rural-Remote
areas is almost always anchored to some form of urban or intermediate
accessibility.

---

### Section 3 — Pipeline A (Reverters)

**Global spatial autocorrelation:**
Global Moran's I = 0.3101 (p = 0.0001, z = 46.25) confirms that demographic
growth in Period B is spatially structured across all municipalities — not
randomly distributed. This result justifies the full spatial analysis and
validates the use of LISA for local cluster detection. The higher I value
relative to earlier estimates reflects the correction of building W on the
full GDF rather than the rural subset only.

**LISA quadrant distribution:**
LL (1,112) dominates across all municipalities, reflecting the spatial
concentration of structural depopulation across the rural interior. HH (844)
identifies statistically robust growth clusters. Within the rural stratum,
HH is more frequent in Rural-Accessible (446; 11.5% of typology) than
Rural-Remote (102; 3.6%), consistent with the peri-urban hypothesis.
HL (188 total; Rural-Accessible: 75 (1.9%), Rural-Remote: 109 (3.8%))
identifies spatial outliers — municipalities with above-average growth
surrounded by below-average-growth neighbours — the strongest statistical
signal for neo-rural hotspots. HL is proportionally more common in
Rural-Remote than Rural-Accessible, confirming that isolated growth in a
declining rural context is concentrated in the more peripheral typology.

**Extended clusters — methodological note:**
The extended cluster analysis collapses 1,689 of the 2,090 reverters into
a single supercluster, reflecting the high spatial connectivity of rural Spain
when shared-neighbour proximity is used as the linking criterion. This confirms
that extended clusters are not analytically useful at the national scale.
The direct cluster analysis is the operative spatial classification for the
paper. Extended clusters are retained in the GeoPackage for reference but
not interpreted further.

**Reverter spatial classification:**
The consolidated spatial classification reveals that the vast majority of
reverters belong to direct contiguity clusters: 1,079 in Rural-Accessible
(83.3%) and 674 in Rural-Remote (84.9%). True spatial outliers are a small
minority: 14 in Rural-Accessible and 13 in Rural-Remote.

The peri-urban signal dominates: of the 1,753 reverters in direct clusters,
the majority are within 60 km of an urban centre, concentrated in large clusters
that are predominantly near urban. Only a minority of direct-cluster reverters
lie beyond this threshold, concentrated in Rural-Remote.

Among large clusters (>50 municipalities), four of five are dominated by
near urban municipalities, confirming their peri-urban character. The exception
is the Aragón cluster (n=52, Huesca and Zaragoza provinces, 43 near
intermediate, 9 near urban), which represents demographic recovery anchored
around intermediate comarcal centres rather than large urban areas — a
structurally distinct pattern from metropolitan overflow.

The top reverters in Rural-Remote show two distinct patterns: (1) Lozoya-
Somosierra municipalities (La Hiruela, Horcajo de la Sierra-Aoslos, Puebla
de la Sierra, Robregordo, Madarcos — Madrid province, within 60 km) with
strong growth driven by peri-urbanisation; and (2) genuinely remote
municipalities in Soria, Guadalajara interior, Aragón and Extremadura beyond
60 km of any urban centre. Fuente el Olmo de Fuentidueña (Segovia,
dist. = 71.9 km, LISA HL) is a notable isolated case with LISA HL
confirmation — a strong neo-rural signal in the reverter pipeline.

Finestrat (Alicante, Rural-Accessible, pop. ref. = 7,103, +55.4%) appears in
the Rural-Accessible top 30 as an outlier driven by coastal tourism and
residential growth rather than counter-urbanisation. This confirms that
positive cumulative growth in Period B does not discriminate between growth
types; the socioeconomic analysis (SIDAMUN variables) is required to
distinguish permanent residential growth from tourism-driven or seasonal
dynamics.

---

### Section 4 — Pipeline B (Sustained dynamisers)

**Dynamiser counts and spatial classification:**
Rural-Remote dynamisers are few (n=127, 4.5% of Rural-Remote municipalities)
but analytically significant — they maintained positive growth across both
periods even during the pre-inflection decline that affected most of rural
Spain. Rural-Accessible dynamisers are more numerous (n=626, 16.1%) and
predominantly near urban (624 of 626), confirming that sustained demographic
resilience in accessible rural areas is strongly associated with urban proximity.

Spatial outliers are proportionally more common among Rural-Remote reverters
(47.6%) than among Rural-Remote dynamisers (34.6%), suggesting that
post-2018 demographic recovery in peripheral areas tends to occur in greater
geographic isolation than structurally sustained growth — which more frequently
forms spatially contiguous clusters with other growing municipalities. Among
Rural-Accessible dynamisers the figure is 12.0%. The 19 Rural-Remote spatial
outlier dynamisers are strong candidates for structural neo-rural resilience:
Hombrados (Guadalajara, dist. = 74.1 km, +51.7%) and Gargüera (Cáceres,
dist. = 85.6 km, +46.1%) stand out as the most analytically pure cases —
remote, isolated growth sustained across both periods.

Dynamiser clusters are much smaller than reverter clusters (maximum n=36
vs. n=182), confirming that structurally sustained rural growth is a
localised phenomenon without the regional-scale spatial coherence seen
in the reverter pipeline.

**Notable outliers:**
Yebes (Guadalajara, Rural-Accessible, pop. ref. = 4,189, +60.2% in Period B)
is driven by residential expansion in the Madrid-Guadalajara corridor.
Antigua (Las Palmas, Canarias, pop. ref. = 12,972) represents insular
tourist-residential dynamics. Both are retained in the dataset but flagged
for exclusion from the neo-rural interpretation; the socioeconomic analysis
(SIDAMUN) will provide the systematic filter.

**LISA cross-reference:**
Of 753 dynamisers, 204 are LISA HH (Rural-Accessible: 183, Rural-Remote: 21)
and 12 are LISA HL (Rural-Accessible: 7, Rural-Remote: 5). The LISA HL
dynamisers in Rural-Remote — municipalities sustaining growth across both
periods while surrounded by declining neighbours, statistically significant —
are the most rigorous empirical definition of structural neo-rural resilience
available with these data.

---

### Priority fieldwork candidates (combined pipelines)

The intersection of remote geography (beyond both proximity thresholds, or
dist. to nearest urban centre > 80 km), positive growth in Period B, and LISA HL
or spatial outlier classification defines the highest-priority municipalities
for qualitative research:

**Reverters — Mainland remote:**
Serranía Celtibérica cluster in Guadalajara (including Hombrados +51.7%,
Morenilla +39.5%, Riba de Saelices +29.9%, Anquela del Pedregal +21.7%);
Fuente el Olmo de Fuentidueña (Segovia, dist. = 71.9 km, LISA HL);
La Zoma (Teruel, dist. = 99.3 km); Valdemadera (La Rioja, dist. = 61.7 km,
LISA HL).

**Dynamisers — Mainland remote spatial outliers:**
Hombrados (Guadalajara, dist. = 74.1 km, +51.7%), Gargüera (Cáceres,
dist. = 85.6 km, +46.1%), Villanueva de Azoague (Zamora, dist. = 51.8 km,
LISA HL), El Frago (Zaragoza, dist. = 66.9 km, +30.3%).

These municipalities combine the three strongest criteria for genuine
counter-urbanisation: Goerlich Rural-Remote typology, proximity beyond
60 km from any urban or intermediate centre, and demographic growth against
the regional trend confirmed by spatial statistics.

In [ ]:
print(gdf_rural[
    (gdf_rural['tipo_goerlich'] == 'Rural - Remoto') &
    (gdf_rural['lisa_sig_quad'] == 'HL')
].groupby(['CCAA_Name', 'Prov_Name']).size()
 .sort_values(ascending=False)
 .head(15))